# Transform Product Catalog
Transforms the product catalog data and saves the cleaned result as a Silver table.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [0]:
df = spark.read.table("ecommerce_catalog.bronze.product_catalog_raw")
display(df)

In [0]:
df = df.withColumn(
    "is_imputed", 
    F.lit(None).cast("string")
    )

target_column_order = df.columns

In [0]:
df = df.withColumn("category", F.initcap(F.col("category")))

display(df)

In [0]:
df = df.withColumn("active", F.lower(F.col("active")))

display(df)


In [0]:
display(df.filter(F.col("manufacturer").isNull()))

In [0]:
display(df.filter(F.col("subcategory").isNull()))

In [0]:
display(df.filter(F.col("category") == "Tools & Home Improvement"))



### **Observations**
- The `subcategory` cannot be unambiguously determined, as the rotary hammers are not mapped to a consistent `subcategory`.
- `manufacturer` can be inferred from the product name, as it corresponds to the first word of `product_name` in each case.

In [0]:

product = (
    df.filter(F.col("manufacturer").isNull())
      .select("product_name")
      .first()[0]
)
manufacturer_of_product = product.split()[0]

condition = F.col("manufacturer").isNull()

df = df \
    .withColumn(
        "is_imputed",
        F.when(condition, F.lit("manufacturer"))
         .otherwise(F.col("is_imputed"))  # Behält vorhandene Werte, falls die Spalte schon existiert
    ) \
    .withColumn(
        "manufacturer",
        F.when(condition, F.lit(manufacturer_of_product))
         .otherwise(F.col("manufacturer"))
    ) 
    

display(df.filter(F.col("manufacturer") == manufacturer_of_product))

In [0]:
display(df.filter(F.col("stock_quantity") > 1000))

### **Anomalous Stock Level (stock_quantity)**
An exceptionally high stock level of 999999 was identified for product `P0104`. This value deviates significantly from the stock levels of the remaining products and is therefore classified as potentially erroneous.

However, because no additional inventory or warehouse data is available, it cannot be definitively determined whether the value is genuinely corrupt, nor can a correct replacement value be reliably established. The value is therefore retained unchanged in the Silver layer, and the anomaly is documented.

In [0]:
display(df.filter(F.col("product_id")=="P0022"))

In [0]:
condition_negativ = F.col("unit_cost") < 0

df = df \
    .withColumn(
        "is_imputed",
    
        F.when(
            condition_negativ,
            F.when(F.col("is_imputed").isNotNull(), F.concat_ws(", ", F.col("is_imputed"), F.lit("unit_cost")))
             .otherwise(F.lit("unit_cost"))
        ).otherwise(F.col("is_imputed"))
    ) \
    .withColumn(
        "unit_cost",
        F.when(condition_negativ, F.lit(None))
         .otherwise(F.col("unit_cost"))
    ) 

display(df.filter(F.col("product_id")=="P0022"))



### **Negative Purchase Price (unit_cost)**
A negative `unit_cost` was identified for one product. Because a reliable purchase price cannot be reconstructed from the available data, the invalid value was not replaced with an estimated figure, but instead set to `NULL` in the Silver layer. This avoids introducing unsubstantiated assumptions into the cleaned dataset.

In [0]:
sparks = SparkSession.builder.getOrCreate()

if sparks.catalog.tableExists("ecommerce_catalog.silver.silver_products"):
   dlt_obj = DeltaTable.forName(spark, "ecommerce_catalog.silver.silver_products")

   dlt_obj.alias("trg").merge(df.alias("src"),"trg.product_id == src.product_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df.write.format("delta")\
        .mode("append")\
        .saveAsTable("ecommerce_catalog.silver.silver_products")

In [0]:
%sql
SELECT * 
FROM ecommerce_catalog.silver.silver_products;

### **Summary of Silver Transformation**
The product data was cleaned and standardized for the Silver layer.

The following transformations and data quality decisions were implemented:
- Inconsistent casing in `active` was unified.
- Inconsistent casing in `category` was standardized.
- The missing value in `manufacturer` was unambiguously reconstructed and populated based on `product_name`.
- The missing value in `subcategory` was investigated. Because an unambiguous mapping could not be determined from comparable products, the `NULL` value was intentionally retained.
- The anomalously high value of 999999 in `stock_quantity` was investigated. Since no additional inventory data is available for validation or reliable reconstruction, the value was retained unchanged and the anomaly was documented.
- The negative `unit_cost` for `P0022` was classified as domain-invalid. Because no reliable purchase price could be reconstructed from the existing data, the value was set to `NULL`.

The cleaned and standardized product data was subsequently persisted as a Silver table.